# American option simulation, aimed to solve unregular repayment issue of non standard debt

XU YIFEI

ZSAMC_RI

Non standard debts, which is secondary business line of AMC, have some non standard attributions compared with standard debt traded within exchange. Typically, enterprises rise non standard debt from AMC because they cannot fund capital from banks and open market. AMC has comparasion disadvantage with banks and open market, thus enterprises always have motivation to switch back, and early repay AMC non standard debt. To simulate the early repayment, the notebook would use american option dynamic because the american option can also early exercise when the price comes to a 'suitable' regime.

American option early repay decision is based on whether the immediate exercise is greater than continuous value, which is the hold to maturity discounted payoff.

## Import packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import xgboost as xgb

## Forward simulation

In [2]:

# simulate heston model paths
def Heston_model_paths(params, S0, r, T, num_paths):
    kappa, theta, sigma, rho, v0 = params
    # dt is per day, or 1/365 year
    dt = 1 / 365
    num_steps = int(T * 365)
    S = np.zeros((num_steps + 1, num_paths))
    v = np.zeros((num_steps + 1, num_paths))
    S[0] = S0
    v[0] = v0
    for t in range(1, num_steps + 1):
        z1 = np.random.normal(size=num_paths)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.normal(size=num_paths)
        v[t] = np.maximum(v[t-1] + kappa * (theta - v[t-1]) * dt + sigma * np.sqrt(v[t-1] * dt) * z1, 0)
        S[t] = S[t-1] * np.exp((r - 0.5 * v[t-1]) * dt + np.sqrt(v[t-1] * dt) * z2)
    return S

def Bates_model_paths(params, S0, r, T, num_paths):
    kappa, theta, sigma, rho, v0, muJ, sigmaJ, lamb = params
    dt = 1 / 365
    num_steps = int(T * 365)
    S = np.zeros((num_steps + 1, num_paths))
    v = np.zeros((num_steps + 1, num_paths))
    S[0] = S0
    v[0] = v0
    for t in range(1, num_steps + 1):
        z1 = np.random.normal(size=num_paths)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.normal(size=num_paths)
        J = np.random.poisson(lamb * dt, num_paths) * (np.exp(muJ + sigmaJ * np.random.normal(size=num_paths)) - 1)
        v[t] = np.maximum(v[t-1] + kappa * (theta - v[t-1]) * dt + sigma * np.sqrt(v[t-1] * dt) * z1, 0)
        S[t] = S[t-1] * np.exp((r - 0.5 * v[t-1]) * dt + np.sqrt(v[t-1] * dt) * z2) * (1 + J)
    return S


## Parameters

In [3]:
# Parameters
np.random.seed(76)
S0 = 100  # Initial stock price
K = 80  # Strike price
r = 0.05  # Risk-free rate
T = 1.0  # Time to maturity
sigma = 0.2  # Volatility
N = 1000  # Number of paths
params_Heston = (0.1, 0.1, 0.1, -0.5, 0.1)
params_Bates = (0.60218861, 0.23021339, 2.96595818, -0.89633531, 0.02514128, -0.34793572, 0.97376232, 0.15454091)

## Trail output

In [4]:

S_paths = Bates_model_paths(params_Bates, S0, r, T, N) # rows are time steps, columns are paths
payoff = np.maximum(S_paths - K, 0)
n_steps, n_paths = S_paths.shape
dt = T / n_steps

## Backward induction

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from xgboost import XGBRegressor
def backward_induction(S_paths, payoff, r, T, method):
    V = np.zeros_like(S_paths)
    V[-1] = payoff[-1]

    for t in range(n_steps-2, -1, -1):
        itm = payoff[t] > 0
        X = S_paths[t, itm]
        y = np.exp(-r * dt) * V[t+1, itm]
        
        if len(X) == 0:
            continue

        if method == 'linear':
            poly = PolynomialFeatures(degree=2)
            X_poly = poly.fit_transform(X.reshape(-1, 1))
            reg = LinearRegression().fit(X_poly, y)
            continuous_values = reg.predict(X_poly)
        elif method == 'xgboost':
            reg = XGBRegressor().fit(X.reshape(-1, 1), y)
            continuous_values = reg.predict(X.reshape(-1, 1))
        else:
            raise ValueError('Invalid method')
        
        exercise = payoff[t, itm] > continuous_values
        V[t, itm] = np.where(exercise, payoff[t, itm], np.exp(-r * dt) * V[t+1, itm])
    option_price = np.mean(np.exp(-r * dt) * V[0])
    return option_price

In [6]:
option_price_linear = backward_induction(S_paths, payoff, r, T, 'linear')
print(f"Option price using linear regression: {option_price_linear}")
option_price_xgb = backward_induction(S_paths, payoff, r, T, 'xgboost')
print(f"Option price using XGBoost: {option_price_xgb}")

Option price using linear regression: 28.116466202995415
Option price using XGBoost: 42.214508941502594


## Further design the regression algorithm to calculate continous value

## LSTM

In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm  # Import tqdm for progress bars

# Normalize the data
scaler = MinMaxScaler()
S_paths_scaled = scaler.fit_transform(S_paths.T).T

# Define LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size=2, hidden_layer_size=100, output_size=1):
        super(LSTMModel, self).__init__()
        self.hidden_layer_size = hidden_layer_size
        self.lstm = nn.LSTM(input_size, hidden_layer_size)
        self.linear = nn.Linear(hidden_layer_size, output_size)
        self.hidden_cell = (torch.zeros(1, 1, self.hidden_layer_size),
                            torch.zeros(1, 1, self.hidden_layer_size))

    def forward(self, input_seq):
        lstm_out, self.hidden_cell = self.lstm(input_seq, self.hidden_cell)
        predictions = self.linear(lstm_out[-1])
        return predictions

# Train LSTM model
def train_lstm_model(S_paths, payoff, r, dt, epochs=100):
    model = LSTMModel(input_size=2)  # Ensure input_size matches the input tensor's last dimension
    loss_function = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in tqdm(range(epochs), desc="Epochs"):  # Add tqdm for epoch loop
        for t in tqdm(range(1, S_paths.shape[0]), desc="Time Steps", leave=False):  # Add tqdm for time step loop
            X = torch.tensor(S_paths[t-1:t+1].T, dtype=torch.float32).view(-1, 1, 2)  # Shape: (batch_size, seq_len, input_size)
            y = torch.tensor(np.exp(-r * dt) * payoff[t], dtype=torch.float32).view(-1, 1)
            
            optimizer.zero_grad()
            model.hidden_cell = (torch.zeros(1, 1, model.hidden_layer_size),
                                 torch.zeros(1, 1, model.hidden_layer_size))
            
            y_pred = model(X)
            single_loss = loss_function(y_pred, y)
            single_loss.backward()
            optimizer.step()
    
    return model

# Longstaff-Schwartz Algorithm with LSTM
def backward_induction_lstm(S_paths, payoff, r, T, model):
    n_steps, n_paths = S_paths.shape
    dt = T / n_steps
    V = np.zeros_like(S_paths)
    V[-1] = payoff[-1]

    for t in tqdm(range(n_steps-2, -1, -1), desc="Backward Induction"):  # Add tqdm for backward induction loop
        itm = payoff[t] > 0
        X = torch.tensor(S_paths[t:t+2, itm].T, dtype=torch.float32).view(-1, 1, 2)
        
        if len(X) == 0:
            continue
        
        model.hidden_cell = (torch.zeros(1, 1, model.hidden_layer_size),
                             torch.zeros(1, 1, model.hidden_layer_size))
        continuation_value = model(X).detach().numpy()
        
        exercise = payoff[t, itm] > continuation_value
        V[t, itm] = np.where(exercise, payoff[t, itm], np.exp(-r * dt) * V[t+1, itm])
    
    option_price = np.mean(np.exp(-r * dt) * V[0])
    return option_price

# Train the LSTM model
lstm_model = train_lstm_model(S_paths_scaled, payoff, r, dt)

# Calculate option price using LSTM
price_lstm = backward_induction_lstm(S_paths_scaled, payoff, r, T, lstm_model)

print(f"Option price using LSTM: {price_lstm:.2f}")

Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/Users/nannan/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([1000, 1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Backward Induction: 100%|██████████| 365/365 [00:11<00:00, 32.74it/s]

Option price using LSTM: 21.69


## GRU

In [8]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm  # Import tqdm for progress bars

# Normalize the data
scaler = MinMaxScaler()
S_paths_scaled = scaler.fit_transform(S_paths.T).T

# Define GRU model
class GRUModel(nn.Module):
    def __init__(self, input_size=2, hidden_layer_size=100, output_size=1):
        super(GRUModel, self).__init__()
        self.hidden_layer_size = hidden_layer_size
        self.gru = nn.GRU(input_size, hidden_layer_size)
        self.linear = nn.Linear(hidden_layer_size, output_size)
        self.hidden_cell = torch.zeros(1, 1, self.hidden_layer_size)

    def forward(self, input_seq):
        gru_out, self.hidden_cell = self.gru(input_seq, self.hidden_cell)
        predictions = self.linear(gru_out[-1])
        return predictions

# Train GRU model
def train_gru_model(S_paths, payoff, r, dt, epochs=100):
    model = GRUModel(input_size=2)  # Ensure input_size matches the input tensor's last dimension
    loss_function = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in tqdm(range(epochs), desc="Epochs"):  # Add tqdm for epoch loop
        for t in tqdm(range(1, S_paths.shape[0]), desc="Time Steps", leave=False):  # Add tqdm for time step loop
            X = torch.tensor(S_paths[t-1:t+1].T, dtype=torch.float32).view(-1, 1, 2)  # Shape: (batch_size, seq_len, input_size)
            y = torch.tensor(np.exp(-r * dt) * payoff[t], dtype=torch.float32).view(-1, 1)
            
            optimizer.zero_grad()
            model.hidden_cell = torch.zeros(1, 1, model.hidden_layer_size)
            
            y_pred = model(X)
            single_loss = loss_function(y_pred, y)
            single_loss.backward()
            optimizer.step()
    
    return model

# Longstaff-Schwartz Algorithm with GRU
def backward_induction_gru(S_paths, payoff, r, T, model):
    n_steps, n_paths = S_paths.shape
    dt = T / n_steps
    V = np.zeros_like(S_paths)
    V[-1] = payoff[-1]

    for t in tqdm(range(n_steps-2, -1, -1), desc="Backward Induction"):  # Add tqdm for backward induction loop
        itm = payoff[t] > 0
        X = torch.tensor(S_paths[t:t+2, itm].T, dtype=torch.float32).view(-1, 1, 2)
        
        if len(X) == 0:
            continue
        
        model.hidden_cell = torch.zeros(1, 1, model.hidden_layer_size)
        continuation_value = model(X).detach().numpy()
        
        exercise = payoff[t, itm] > continuation_value
        V[t, itm] = np.where(exercise, payoff[t, itm], np.exp(-r * dt) * V[t+1, itm])
    
    option_price = np.mean(np.exp(-r * dt) * V[0])
    return option_price

# Train the GRU model
gru_model = train_gru_model(S_paths_scaled, payoff, r, dt)

# Calculate option price using GRU
price_gru = backward_induction_gru(S_paths_scaled, payoff, r, T, gru_model)

print(f"Option price using GRU: {price_gru:.2f}")

Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/Users/nannan/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([1000, 1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Backward Induction: 100%|██████████| 365/365 [00:12<00:00, 29.51it/s]

Option price using GRU: 20.00


In [ ]:
def call_to_spread(call, S, K, T, r):
    